# 01 — Land sample data

Copies bundle-synced CSVs from `fixtures/sample-data` into the UC Volume landing path.
Runs on the Shared all-purpose cluster.

### Step 1 — Configure paths

Read widgets for the catalog, Unity Catalog Volume landing path, and workspace `source_path` where the bundle-synced fixture CSVs live. These paths drive the copy in the next step.

In [ ]:
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("landing_path", "/Volumes/actuarial/ml/landing")
dbutils.widgets.text("source_path", "")

catalog = dbutils.widgets.get("catalog")
landing_path = dbutils.widgets.get("landing_path").rstrip("/")
source_path = dbutils.widgets.get("source_path").rstrip("/")

spark.sql(f"USE CATALOG `{catalog}`")
print(f"source_path={source_path}")
print(f"landing_path={landing_path}")

### Step 2 — Copy fixtures into the volume

Map each sample CSV into a landing subdirectory (`claims`, `premiums`, `risk_zones`, `cyclone_events`). Resolve `/Workspace/...` if the raw `source_path` is not found, then copy files into the UC Volume so later notebooks can read a stable landing location.

In [ ]:
import shutil
from pathlib import Path

file_map = {
    "claims_bordereau.csv": "claims",
    "premium_bordereau.csv": "premiums",
    "risk_zone_lookup.csv": "risk_zones",
    "cyclone_events.csv": "cyclone_events",
}

src_root = Path(source_path)
if not src_root.exists():
    alt = Path("/Workspace") / source_path.lstrip("/")
    if alt.exists():
        src_root = alt

if not src_root.exists():
    raise FileNotFoundError(f"source_path not found: {source_path}")

for filename, subdir in file_map.items():
    src = src_root / filename
    if not src.exists():
        raise FileNotFoundError(f"Missing sample file: {src}")

    dest_dir_uri = f"{landing_path}/{subdir}"
    dbutils.fs.mkdirs(dest_dir_uri)
    dest = Path(landing_path) / subdir / filename
    shutil.copyfile(src, dest)
    print(f"Landed {src} -> {dest}")

print("Landing complete.")